# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

The dataset fields can be categorized into four groups based on their role in analysis. The label is clicks, as it represents the outcome we want to predict. The features include impressions, ctr, and avg_position, which are directly related to search performance and influence clicks. The context fields include url, start_date, and end_date, which help define the structure and grouping of the data but are not used directly as predictors. The excluded fields include identifiers, redundant columns, and any variables that could introduce data leakage (such as future metrics), as well as unstructured data that cannot be directly used in modeling.

In [11]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

grain_cols = ["content_id"]

dup = (
    df.groupby(grain_cols)
    .size()
    .reset_index(name="count")
)

duplicates = dup[dup["count"] > 1]

print("Duplicate rows at grain level:")
print(duplicates)

print("\nTotal rows:", len(df))
print("Unique content_id rows:", df["content_id"].nunique())

Duplicate rows at grain level:
Empty DataFrame
Columns: [content_id, count]
Index: []

Total rows: 30000
Unique content_id rows: 30000


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

The dataset fields can be categorized into four groups based on their role in analysis. The label is clicks, as it represents the outcome we want to predict. The features include impressions, ctr, and avg_position, which are directly related to search performance and influence clicks. The context fields include url, start_date, and end_date, which help define the structure and grouping of the data but are not used directly as predictors. The excluded fields include identifiers, redundant columns, and any variables that could introduce data leakage (such as future metrics), as well as unstructured data that cannot be directly used in modeling.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Inspect columns
print("Columns in dataset:")
print(df.columns.tolist())

# Example grouping (adjust if dataset has more fields)
label = ["clicks"]

features = [col for col in df.columns if col in ["impressions", "ctr", "avg_position"]]

context = [col for col in df.columns if col in ["url", "start_date", "end_date"]]

excluded = [col for col in df.columns if col not in label + features + context]

print("\nLabel:", label)
print("Features:", features)
print("Context:", context)
print("Excluded:", excluded)

Columns in dataset:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Label: ['clicks']
Features: ['ctr', 'avg_position']
Context: []
Excluded: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', '

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

All assumptions about the dataset must be verified using data checks. First, we confirm that each (url, start_date, end_date) combination is unique to validate the unit of analysis. Next, we check for missing values in key columns like clicks and impressions to ensure data quality. We also verify that the date range is valid by ensuring that start_date is earlier than end_date. Additionally, we compute the duration of each time window to check consistency and examine summary statistics to identify anomalies or outliers in the data.

In [13]:

print("Rows:", len(df))
print("Unique content_id:", df["content_id"].nunique())

print("\nMissing values:")
print(df.isnull().sum().sort_values(ascending=False).head(10))
print("\nCompare last vs previous 30d:")
print(df[["clicks_last_30d", "clicks_prev_30d"]].describe())

numeric_cols = df.select_dtypes(include="number").columns

negatives = {}
for col in numeric_cols:
    negatives[col] = (df[col] < 0).sum()

neg_df = pd.Series(negatives)
print("\nColumns with negative values:")
print(neg_df[neg_df > 0])

print("\nTrend distribution:")
print(df["trend_direction"].value_counts())

print("\nTrend percentage stats:")
print(df["trend_pct"].describe())

Rows: 30000
Unique content_id: 30000

Missing values:
provider_used        21438
word_count            7699
char_count            7699
word_count_tier       7699
char_count_tier       7699
model_used            5733
trend_pct             3388
competition_level     2610
search_volume         2468
cpc                   2468
dtype: int64

Compare last vs previous 30d:
       clicks_last_30d  clicks_prev_30d
count     30000.000000     30000.000000
mean          4.933867         5.435100
std          23.929393        28.358673
min           0.000000         0.000000
25%           0.000000         0.000000
50%           0.000000         0.000000
75%           2.000000         2.000000
max        1176.000000      1627.000000

Columns with negative values:
trend_pct    19715
dtype: int64

Trend distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Trend percentage stats:
count    26612.000000
mean        -4.7859

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset has several limitations that restrict the conclusions we can draw from it. First, it cannot establish causality; it only shows relationships between variables, not whether one causes another. Second, it lacks external context such as search algorithm updates, seasonal trends, and competitor activity, all of which can influence performance. Third, the dataset may be unbalanced, with some URLs having more observations than others, which can bias analysis. Additionally, since the data comes from Google Search Console, it may include sampling issues and missing low-volume data. The dataset also does not include user behavior metrics such as engagement or conversions, nor does it capture content quality, limiting deeper insights. Finally, overlapping time windows may introduce double counting if not handled carefully.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
corr = df[[
    "clicks_last_30d",
    "clicks_prev_30d",
    "clicks_90d"
]].corr()

print("Correlation between time windows:")
print(corr)


print("\nTrend imbalance:")
print(df["trend_direction"].value_counts(normalize=True))

print("\nContent age stats:")
print(df["content_age_days"].describe())

Correlation between time windows:
                 clicks_last_30d  clicks_prev_30d  clicks_90d
clicks_last_30d         1.000000         0.922519    0.946757
clicks_prev_30d         0.922519         1.000000    0.976504
clicks_90d              0.946757         0.976504    1.000000

Trend imbalance:
trend_direction
down      0.542067
stable    0.198733
up        0.146267
new       0.074533
flat      0.038400
Name: proportion, dtype: float64

Content age stats:
count    30000.00000
mean       256.16780
std        132.70793
min         90.00000
25%        132.00000
50%        236.00000
75%        333.00000
max        564.00000
Name: content_age_days, dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.